# Manual: Dirección de Rechazo en LLMs con Arquitectura MoE

## Referencia: "Refusal in Language Models Is Mediated by a Single Direction" (arXiv:2406.11717)

---

### Versión MoE con PyTorch Nativo (sin TransformerLens)

TransformerLens no soporta modelos MoE pequeños. Este notebook usa **hooks nativos de PyTorch**
para implementar la misma metodología directamente con HuggingFace Transformers.

#### Modelo: OLMoE-1B-7B-0924-Instruct (Allen AI)
- **7B params totales**, solo **1B activo** por token
- 64 expertos, Top-8 seleccionados por token
- SIN shared expert (todos routing)
- ~4GB en 4-bit → **cabe de sobra en T4 gratuita**

#### Arquitectura OLMoE

```
Capa i del Decoder (16 capas):
  resid ─→ RMSNorm ─→ Self-Attention ─→ + ─→ RMSNorm ─→ MoE ─→ + ─→ resid
   │                                    │                        │
   └───────── residual ─────────────────┘                        │
   └───────────────────── residual ─────────────────────────────┘

  MoE Layer (OlmoeSparseMoeBlock):
    ┌── Router (gate) ──→ Top-8 de 64 expertos
    │
    │   ┌─────────────────────────────────────────────┐
    │   │ Expert_k:                                   │
    │   │   gate_up = gate_up_proj[k](x)              │
    │   │   output = down_proj[k](SiLU(gate) * up)    │
    │   └─────────────────────────────────────────────┘
    │
    └── Weighted sum of 8 expert outputs → residual
```

#### Estructura de Módulos PyTorch

```python
model.model.layers[i]                        # Capa completa
model.model.layers[i].self_attn.o_proj       # Salida de atención
model.model.layers[i].mlp                    # Bloque MoE
model.model.layers[i].mlp.gate               # Router (OlmoeTopKRouter)
model.model.layers[i].mlp.experts            # OlmoeExperts (tensor 3D)
model.model.layers[i].mlp.experts.down_proj  # (64, d_model, intermediate)
```

**Nota**: Los expertos se almacenan como un **tensor 3D** (64 x d_model x intermediate),
NO como módulos individuales. Esto es más eficiente pero cambia cómo se hace la ortogonalización.

---
## 1. Setup

**NO usamos TransformerLens** - usamos HuggingFace Transformers + PyTorch directamente.

Ejecutar esta celda y **reiniciar el runtime** antes de continuar.

In [ ]:
%%capture
!pip install transformers accelerate bitsandbytes
!pip install numpy==1.26.4 --force-reinstall
!pip install scikit-learn pandas --force-reinstall
!pip install colorama

In [ ]:
import torch
import functools
import requests
import pandas as pd
import io
import textwrap
import gc

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from colorama import Fore

print("Imports completados correctamente")
print("(Sin TransformerLens - usando PyTorch nativo)")

---
## 2. Carga del Modelo MoE

| Parámetro | Valor para OLMoE |
|-----------|------------------|
| `MODEL_PATH` | `allenai/OLMoE-1B-7B-0924-Instruct` |
| `LAYER` | 8 (~50% de 16 capas) |
| `POS` | -1 (último token) |
| `USE_4BIT` | True (solo ~4GB VRAM) |

| Modelo | Params | Activos | Capas | Expertos | VRAM fp16 | VRAM 4bit |
|--------|--------|---------|-------|----------|-----------|----------|
| **OLMoE-1B-7B** | **7B** | **1B** | **16** | **64 (Top-8)** | **~14GB** | **~4GB** |
| Qwen1.5-MoE-A2.7B | 14.3B | 2.7B | 24 | 60+1 shared | ~28GB | ~8GB |
| Mixtral-8x7B | 47B | 13B | 32 | 8 | ~90GB | ~24GB |

In [ ]:
# ============================================================================
# CONFIGURACIÓN DEL MODELO MoE
# ============================================================================

MODEL_PATH = 'allenai/OLMoE-1B-7B-0924-Instruct'  # <-- MODIFICAR AQUÍ

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# OLMoE cabe en T4 incluso en fp16 (~14GB), pero 4-bit es más seguro
USE_4BIT = True  # <-- True para T4, False si tienes >16GB VRAM

# Capa para extracción (~50% de 16 capas)
LAYER = 8  # <-- MODIFICAR SEGÚN MODELO

POS = -1  # Último token

N_INST_TRAIN = 32
N_INST_TEST = 32

print(f"Configuración MoE:")
print(f"  Modelo: {MODEL_PATH}")
print(f"  Dispositivo: {DEVICE}")
print(f"  Cuantización 4-bit: {USE_4BIT}")
print(f"  Capa: {LAYER}")
print(f"  Posición: {POS}")

In [ ]:
# ============================================================================
# CARGA DEL MODELO MoE
# ============================================================================
# OLMoE está soportado nativamente en transformers (>=4.44)
# NO necesita trust_remote_code

print(f"Cargando modelo MoE: {MODEL_PATH}")
print("Esto puede tardar varios minutos...")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Modelo
if USE_4BIT:
    print("  Usando cuantización 4-bit (~4GB VRAM)")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=quantization_config,
        device_map="auto",
        low_cpu_mem_usage=True
    )
else:
    print("  Cargando en float16 (~14GB VRAM)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto"
    )

model.eval()

# Información del modelo
n_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
n_experts = getattr(model.config, 'num_experts', 'N/A')
experts_per_tok = getattr(model.config, 'num_experts_per_tok', 'N/A')

print(f"\nModelo MoE cargado!")
print(f"  Capas: {n_layers}")
print(f"  d_model: {d_model}")
print(f"  Expertos totales: {n_experts}")
print(f"  Expertos por token: {experts_per_tok}")
print(f"  Cuantizado: {USE_4BIT}")

---
## 3. Chat Template

OLMoE usa un formato propio con tokens `<|system|>`, `<|user|>`, `<|assistant|>`.

Usamos `tokenizer.apply_chat_template()` para manejar esto automáticamente.

In [ ]:
# ============================================================================
# TOKENIZACIÓN CON CHAT TEMPLATE
# ============================================================================
# OLMoE tiene su propio chat template integrado en el tokenizer.
# Usamos apply_chat_template() en vez de un template manual.

def tokenize_instructions(instructions: List[str]):
    """
    Tokeniza instrucciones usando el chat template nativo de OLMoE.
    """
    all_input_ids = []
    
    for inst in instructions:
        messages = [{"role": "user", "content": inst}]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        all_input_ids.append(text)
    
    encoded = tokenizer(
        all_input_ids,
        padding=True,
        truncation=False,
        return_tensors="pt"
    )
    return encoded.input_ids, encoded.attention_mask

# Test
test_ids, test_mask = tokenize_instructions(["Hello, how are you?"])
print(f"Template activo: tokenizer.apply_chat_template()")
print(f"Test tokenización: {test_ids.shape}")
print(f"Texto tokenizado: {tokenizer.decode(test_ids[0])}")

---
## 4. Datasets

In [ ]:
# ============================================================================
# CARGA DE DATASETS
# ============================================================================

def get_harmful_instructions():
    url = 'https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv'
    response = requests.get(url)
    dataset = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
    instructions = dataset['goal'].tolist()
    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test

def get_harmless_instructions():
    hf_path = 'tatsu-lab/alpaca'
    dataset = load_dataset(hf_path)
    instructions = []
    for i in range(len(dataset['train'])):
        if dataset['train'][i]['input'].strip() == '':
            instructions.append(dataset['train'][i]['instruction'])
    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test

print("Cargando datasets...")
harmful_inst_train, harmful_inst_test = get_harmful_instructions()
harmless_inst_train, harmless_inst_test = get_harmless_instructions()

print(f"  Harmful: {len(harmful_inst_train)} train, {len(harmful_inst_test)} test")
print(f"  Harmless: {len(harmless_inst_train)} train, {len(harmless_inst_test)} test")

---
## 5. Funciones de Generación y Hook Manager

In [ ]:
# ============================================================================
# GESTIÓN DE HOOKS DE PYTORCH
# ============================================================================

class HookManager:
    """
    Gestor de hooks de PyTorch.
    Equivalente al context manager model.hooks() de TransformerLens.
    """
    def __init__(self):
        self.handles = []
    
    def register(self, module, hook_fn):
        handle = module.register_forward_hook(hook_fn)
        self.handles.append(handle)
        return handle
    
    def clear(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

print("HookManager definido")

In [ ]:
# ============================================================================
# FUNCIONES DE GENERACIÓN
# ============================================================================

def generate_with_model(
    instructions: List[str],
    max_new_tokens: int = 64,
    batch_size: int = 4
) -> List[str]:
    """
    Genera respuestas. Los hooks registrados se aplican automáticamente
    durante model.generate() ya que este llama forward() internamente.
    """
    generations = []
    
    for i in tqdm(range(0, len(instructions), batch_size)):
        batch = instructions[i:i+batch_size]
        input_ids, attention_mask = tokenize_instructions(batch)
        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
        
        generated_ids = output_ids[:, input_ids.shape[1]:]
        texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        generations.extend(texts)
    
    return generations

print("Funciones de generación definidas")

---
## 6. Extracción de la Dirección de Rechazo

### §2.3 del paper - Difference-in-Means

$$\mathbf{r} = \boldsymbol{\mu}_{harmful} - \boldsymbol{\mu}_{harmless}$$

Usamos `output_hidden_states=True` para obtener las activaciones del residual stream.

```python
outputs = model(input_ids, output_hidden_states=True)
# outputs.hidden_states[LAYER] = resid_pre de capa LAYER
```

In [ ]:
# ============================================================================
# EXTRACCIÓN DE LA DIRECCIÓN DE RECHAZO
# ============================================================================

def get_hidden_states(instructions: List[str], layer: int, pos: int) -> Tensor:
    """
    Obtiene activaciones del residual stream.
    Equivale a: cache['resid_pre', layer][:, pos, :] en TransformerLens
    """
    all_acts = []
    batch_size = 4
    
    for i in range(0, len(instructions), batch_size):
        batch = instructions[i:i+batch_size]
        input_ids, attention_mask = tokenize_instructions(batch)
        input_ids = input_ids.to(model.device)
        attention_mask = attention_mask.to(model.device)
        
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
        
        # hidden_states[0] = embedding output
        # hidden_states[i] = output de capa i-1 = resid_pre de capa i
        hidden = outputs.hidden_states[layer]  # (batch, seq_len, d_model)
        acts = hidden[:, pos, :]  # (batch, d_model)
        all_acts.append(acts.cpu().float())
    
    return torch.cat(all_acts, dim=0)

print("Paso 1: Extrayendo activaciones harmful...")
harmful_acts = get_hidden_states(
    harmful_inst_train[:N_INST_TRAIN], LAYER, POS
)

print("Paso 2: Extrayendo activaciones harmless...")
harmless_acts = get_hidden_states(
    harmless_inst_train[:N_INST_TRAIN], LAYER, POS
)

print(f"\n  Harmful acts shape: {harmful_acts.shape}")
print(f"  Harmless acts shape: {harmless_acts.shape}")

# Dirección de rechazo
harmful_mean = harmful_acts.mean(dim=0)
harmless_mean = harmless_acts.mean(dim=0)

# Sin normalizar (para activation addition)
refusal_dir_unnormalized = (harmful_mean - harmless_mean).to(model.device).to(torch.float16)

# Normalizada (para ablación)
refusal_dir = refusal_dir_unnormalized / refusal_dir_unnormalized.norm()

print(f"\n  ✓ Dirección de rechazo calculada")
print(f"    Shape: {refusal_dir.shape}")
print(f"    Norma original: {refusal_dir_unnormalized.norm().item():.4f}")

del harmful_acts, harmless_acts
gc.collect()
torch.cuda.empty_cache()

---
## 7. Bypass de Rechazo: Ablación Direccional

### §3.1 del paper

$$\mathbf{a}' = \mathbf{a} - (\mathbf{a} \cdot \hat{\mathbf{r}}) \hat{\mathbf{r}}$$

Hook en `model.model.layers[i]` que modifica la salida (hidden_states).

En OLMoE, cada capa retorna una tupla `(hidden_states, self_attn_weights, ...)`.

In [ ]:
# ============================================================================
# HOOK DE ABLACIÓN DIRECCIONAL (PyTorch nativo)
# ============================================================================

def make_ablation_hook(direction: Tensor):
    """
    Crea hook que elimina el componente en 'direction' de las hidden states.
    Implementa: a' = a - (a · r̂) r̂
    """
    def hook(module, input, output):
        # output es tupla: (hidden_states, ...)
        hidden_states = output[0]  # (batch, seq_len, d_model)
        dir_vec = direction.to(hidden_states.device, dtype=hidden_states.dtype)
        
        # Proyección: (a · r̂)
        proj_scalar = (hidden_states * dir_vec).sum(dim=-1, keepdim=True)
        # Restar: a' = a - (a · r̂) r̂
        modified = hidden_states - proj_scalar * dir_vec
        
        return (modified,) + output[1:]
    
    return hook

print("Función de ablación direccional definida")

In [ ]:
# ============================================================================
# BASELINE: Sin intervención
# ============================================================================

print("--- BASELINE: Generando respuestas sin intervención ---")
print("(El modelo debería RECHAZAR instrucciones dañinas)\n")

baseline_generations = generate_with_model(
    harmful_inst_test[:N_INST_TEST],
    max_new_tokens=64,
    batch_size=4
)

In [ ]:
# ============================================================================
# CON ABLACIÓN DIRECCIONAL
# ============================================================================

print("--- ABLACIÓN: Registrando hooks en todas las capas ---\n")

hook_mgr = HookManager()
ablation_hook = make_ablation_hook(refusal_dir)

for i in range(n_layers):
    hook_mgr.register(model.model.layers[i], ablation_hook)

print(f"  Hooks registrados: {len(hook_mgr.handles)} (1 por capa)")
print(f"  Capas: 0 a {n_layers - 1}")

print("\nGenerando respuestas con ablación...")
print("(El modelo debería NO rechazar)\n")

intervention_generations = generate_with_model(
    harmful_inst_test[:N_INST_TEST],
    max_new_tokens=64,
    batch_size=4
)

hook_mgr.clear()
print("\nHooks eliminados")

In [ ]:
# ============================================================================
# RESULTADOS: BYPASS DE RECHAZO
# ============================================================================

print("=" * 80)
print("RESULTADOS MoE: BYPASS DE RECHAZO EN INSTRUCCIONES DAÑINAS")
print("=" * 80)

for i in range(min(5, N_INST_TEST)):
    print(f"\n{'─' * 80}")
    print(f"INSTRUCCIÓN {i+1}: {repr(harmful_inst_test[i])}")
    print(f"{'─' * 80}")

    print(Fore.GREEN + "BASELINE (sin intervención):" + Fore.RESET)
    print(textwrap.fill(repr(baseline_generations[i]), width=100,
        initial_indent='  ', subsequent_indent='  '))

    print(Fore.RED + "\nCON ABLACIÓN:" + Fore.RESET)
    print(textwrap.fill(repr(intervention_generations[i]), width=100,
        initial_indent='  ', subsequent_indent='  '))

---
## 8. Inducción de Rechazo: Activation Addition

### §3.2 del paper

$$\mathbf{x}' = \mathbf{x} + \mathbf{r}$$

Añadimos la dirección de rechazo (sin normalizar) para inducir rechazo
en instrucciones inocuas.

In [ ]:
# ============================================================================
# HOOK DE ACTIVATION ADDITION
# ============================================================================

def make_addition_hook(direction: Tensor, scale: float = 1.0):
    """
    Crea hook que AÑADE la dirección de rechazo.
    Implementa: x' = x + scale * r
    """
    def hook(module, input, output):
        hidden_states = output[0]
        dir_vec = direction.to(hidden_states.device, dtype=hidden_states.dtype)
        modified = hidden_states + scale * dir_vec
        return (modified,) + output[1:]
    
    return hook

print("Función de activation addition definida")

In [ ]:
# ============================================================================
# BASELINE HARMLESS
# ============================================================================

print("--- BASELINE HARMLESS ---\n")

baseline_harmless_generations = generate_with_model(
    harmless_inst_test[:N_INST_TEST],
    max_new_tokens=64,
    batch_size=4
)

In [ ]:
# ============================================================================
# CON ACTIVATION ADDITION (inducir rechazo)
# ============================================================================

print("--- ACTIVATION ADDITION: Induciendo rechazo ---")
print(f"  Hook solo en capa {LAYER}\n")

hook_mgr = HookManager()
addition_hook = make_addition_hook(refusal_dir_unnormalized, scale=1.0)
hook_mgr.register(model.model.layers[LAYER], addition_hook)

induced_refusal_generations = generate_with_model(
    harmless_inst_test[:N_INST_TEST],
    max_new_tokens=64,
    batch_size=4
)

hook_mgr.clear()
print("\nHooks eliminados")

In [ ]:
# ============================================================================
# RESULTADOS: INDUCCIÓN DE RECHAZO
# ============================================================================

print("=" * 80)
print("RESULTADOS MoE: INDUCCIÓN DE RECHAZO EN INSTRUCCIONES INOCUAS")
print("=" * 80)

for i in range(min(5, N_INST_TEST)):
    print(f"\n{'─' * 80}")
    print(f"INSTRUCCIÓN {i+1}: {repr(harmless_inst_test[i])}")
    print(f"{'─' * 80}")

    print(Fore.GREEN + "BASELINE:" + Fore.RESET)
    print(textwrap.fill(repr(baseline_harmless_generations[i]), width=100,
        initial_indent='  ', subsequent_indent='  '))

    print(Fore.MAGENTA + "\nCON ACTIVATION ADDITION (rechazo inducido):" + Fore.RESET)
    print(textwrap.fill(repr(induced_refusal_generations[i]), width=100,
        initial_indent='  ', subsequent_indent='  '))

---
## 9. Ortogonalización de Pesos (MoE)

### ⚠️ Solo disponible SIN cuantización 4-bit

**Diferencia clave con otros MoE:** En OLMoE, los expertos se almacenan como
un **tensor 3D** compartido, no como módulos individuales:

```python
# OLMoE: tensor 3D (64, d_model, intermediate)
model.model.layers[i].mlp.experts.down_proj  # Shape: (64, 2048, 1024)

# Qwen MoE: módulos individuales
model.model.layers[i].mlp.experts[j].down_proj  # Shape: (d_model, intermediate)
```

Para ortogonalizar, iteramos sobre la primera dimensión (cada experto).

In [ ]:
# ============================================================================
# ORTOGONALIZACIÓN (solo sin 4-bit)
# ============================================================================

if USE_4BIT:
    print("⚠️ SALTANDO ORTOGONALIZACIÓN")
    print("  No compatible con cuantización 4-bit.")
    print("  La ablación y activation addition ya demuestran el efecto.")
    print("  Para ortogonalizar: USE_4BIT = False (requiere >14GB VRAM)")
else:
    print("Ortogonalizando pesos del modelo MoE...")
    
    def orthogonalize_matrix(matrix, vec):
        """W' = W - r̂ r̂ᵀ W"""
        v = vec.to(matrix.device, dtype=matrix.dtype)
        proj = (matrix * v).sum(dim=-1, keepdim=True) * v
        return matrix - proj
    
    total_experts = 0
    
    # 1. Embeddings
    print("  1. Ortogonalizando embeddings...")
    model.model.embed_tokens.weight.data = orthogonalize_matrix(
        model.model.embed_tokens.weight, refusal_dir
    )
    
    # 2. Cada capa
    print("  2. Ortogonalizando capas...")
    for i in range(n_layers):
        layer = model.model.layers[i]
        
        # Atención: o_proj
        layer.self_attn.o_proj.weight.data = orthogonalize_matrix(
            layer.self_attn.o_proj.weight, refusal_dir
        )
        
        # MoE: down_proj es tensor 3D (n_experts, d_model, intermediate)
        # Ortogonalizar cada experto (slice de la primera dimensión)
        if hasattr(layer.mlp, 'experts') and hasattr(layer.mlp.experts, 'down_proj'):
            down_proj = layer.mlp.experts.down_proj  # Parameter
            n_exp = down_proj.shape[0]
            for e in range(n_exp):
                down_proj.data[e] = orthogonalize_matrix(
                    down_proj.data[e], refusal_dir
                )
                total_experts += 1
        
        if (i + 1) % 4 == 0:
            print(f"    Completadas {i + 1}/{n_layers} capas...")
    
    print(f"\n✓ Modelo ortogonalizado")
    print(f"  Expertos ortogonalizados: {total_experts}")
    
    print("\nGenerando con modelo ortogonalizado...")
    orthogonalized_generations = generate_with_model(
        harmful_inst_test[:N_INST_TEST], max_new_tokens=64, batch_size=4
    )

---
## 10. Resumen

### Modelo: OLMoE-1B-7B (Allen AI)

| Técnica | Funciona en MoE | Implementación |
|---------|----------------|----------------|
| Extracción dirección | ✅ | `output_hidden_states=True` |
| Ablación direccional | ✅ | `register_forward_hook` en cada capa |
| Activation addition | ✅ | `register_forward_hook` en capa LAYER |
| Ortogonalización | ✅* | Iterar tensor 3D de expertos |

\*Solo sin cuantización 4-bit

### Equivalencia TransformerLens ↔ PyTorch Nativo

| TransformerLens | PyTorch Nativo |
|-----------------|----------------|
| `HookedTransformer.from_pretrained()` | `AutoModelForCausalLM.from_pretrained()` |
| `model.run_with_cache()` | `model(output_hidden_states=True)` |
| `model.hooks(fwd_hooks=...)` | `module.register_forward_hook(fn)` |
| `model.W_E` | `model.model.embed_tokens.weight` |
| `block.mlp.W_out` | `layer.mlp.experts.down_proj[e]` |

In [ ]:
print("\n" + "=" * 80)
print("NOTEBOOK MoE COMPLETADO")
print("=" * 80)
print(f"\nModelo: {MODEL_PATH}")
print(f"Arquitectura: MoE ({n_experts} expertos, Top-{experts_per_tok})")
print(f"Cuantización: {'4-bit (NF4)' if USE_4BIT else 'float16'}")
print(f"\nTécnicas demostradas:")
print(f"  1. ✓ Extracción de dirección de rechazo")
print(f"  2. ✓ Bypass de rechazo (ablación direccional)")
print(f"  3. ✓ Inducción de rechazo (activation addition)")
if not USE_4BIT:
    print(f"  4. ✓ Ortogonalización de pesos (tensor 3D de expertos)")
else:
    print(f"  4. ⊘ Ortogonalización (no disponible con 4-bit)")